In [ ]:
import telebot
import os
import io
import cv2
from PIL import Image
from ultralytics import YOLO
import speech_recognition as sr
from pydub import AudioSegment

In [ ]:
class Bot_imagem_audio:
    def __init__(self, token: str):
        self.bot = telebot.TeleBot(token)
        self.bot.register_message_handler(self.responder_texto, content_types=['text'])
        self.bot.register_message_handler(self.responder_foto, content_types=['photo'])
        self.bot.register_message_handler(self.responder_audio, content_types=['voice', 'audio'])
        self.bot.register_message_handler(self.responder_nao_processado, content_types=['document', 'video', 'video_note', 'sticker', 'animation', 'location', 'contact'])

    def responder_texto(self, mensagem):
        self.responder(mensagem)

    def responder_foto(self, mensagem):
        self.detectar_objetos(mensagem)

    def responder_audio(self, mensagem):
        self.transcrever_audio(mensagem)

    def responder(self, mensagem):
        texto = ("Olá, tudo bem? Sou um bot capaz de identificar objetos em uma imagem e também transcrevo áudios.\n"
                 "Você pode me enviar uma foto para eu detectar os objetos presentes na imagem, ou um áudio para que eu transcreva. Envie algo! Estou pronto!")
        self.bot.reply_to(mensagem, texto)

    def responder_nao_processado(self, mensagem):
        texto = ("Desculpe, eu ainda não sei processar esse tipo de arquivo. \n\n"
                 "Por favor, me envie apenas:\n"
                 "*Fotos* (para detecção de objetos)\n"
                 "*Áudios* (para transcrição em texto)")
        
        self.bot.reply_to(mensagem, texto, parse_mode="Markdown")

    def detectar_objetos(self, mensagem):
        raise NotImplementedError()

    def transcrever_audio(self, mensagem):
        raise NotImplementedError()

    def iniciar(self):
        print("Bot iniciado com sucesso! Aguardando mensagens...")
        self.bot.polling()

In [ ]:
class BotImagem(Bot_imagem_audio):
    def __init__(self, token:str):
        super().__init__(token)
        self.model = YOLO("yolov8n.pt")

    def detectar_objetos(self, mensagem):
        self.bot.reply_to(mensagem, "Baixando e analisando a imagem...")
        
        try:
            file_id = mensagem.photo[-1].file_id
            file_info = self.bot.get_file(file_id)
            downloaded_file = self.bot.download_file(file_info.file_path)
            caminho_img = "temp_imagem.jpg"

            with open(caminho_img, 'wb') as new_file:
                new_file.write(downloaded_file)
    
            resultados = self.model(caminho_img)
            imagem_processada = cv2.imread(caminho_img)
            caixas_detectadas = resultados[0].boxes
    
            for caixa in caixas_detectadas:
                classe_id = int(caixa.cls[0])
                confianca = float(caixa.conf[0])
                nome_do_objeto = resultados[0].names[classe_id]
                x1, y1, x2, y2 = map(int, caixa.xyxy[0])
                cv2.rectangle(imagem_processada, (x1, y1), (x2, y2), (0, 0, 255), 2)
                texto = f"{nome_do_objeto}: {confianca:.2f}"
                cv2.putText(imagem_processada, texto, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
            
            caminho_resultado = "resultado.jpg"
            cv2.imwrite(caminho_resultado, imagem_processada)
            
            with open(caminho_resultado, 'rb') as foto_final:
                self.bot.send_photo(mensagem.chat.id, foto_final, caption="Aqui está a análise da sua imagem!")
                
        except Exception as e:
            self.bot.reply_to(mensagem, f"Ocorreu um erro ao processar a imagem: {e}")

class BotAudio(Bot_imagem_audio): 
    def __init__(self, token:str): 
        super().__init__(token)
        self.recognizer = sr.Recognizer() 

    def transcrever_audio(self, mensagem):
        self.bot.reply_to(mensagem, "Baixando e processando o áudio, aguarde um momento...")
        caminho_ogg = "temp_audio.ogg"
        caminho_wav = "temp_audio.wav"
        
        try:
            if mensagem.voice:
                file_id = mensagem.voice.file_id
            elif mensagem.audio:
                file_id = mensagem.audio.file_id
            else:
                self.bot.reply_to(mensagem, "Não foi possível encontrar o áudio na mensagem.")
                return

            file_info = self.bot.get_file(file_id)
            downloaded_file = self.bot.download_file(file_info.file_path)

            with open(caminho_ogg, 'wb') as new_file:
                new_file.write(downloaded_file)

            audio = AudioSegment.from_file(caminho_ogg)
            audio.export(caminho_wav, format="wav")

            with sr.AudioFile(caminho_wav) as source:
                audio_data = self.recognizer.record(source)
                texto_transcrito = self.recognizer.recognize_google(audio_data, language='pt-BR')

            self.bot.reply_to(mensagem, f"🎙️ *Áudio transcrito:*\n\n{texto_transcrito}", parse_mode="Markdown")

        except sr.UnknownValueError:
            self.bot.reply_to(mensagem, "Desculpe, o áudio estava muito baixo ou não consegui entender o que foi dito.")
        except sr.RequestError as e:
            self.bot.reply_to(mensagem, f"Erro de conexão com o serviço do Google: {e}")
        except Exception as e:
            self.bot.reply_to(mensagem, f"Ocorreu um erro inesperado ao processar o áudio: {e}")
    
        finally:
            if os.path.exists(caminho_ogg):
                os.remove(caminho_ogg)
            if os.path.exists(caminho_wav):
                os.remove(caminho_wav)

class BotTelegram(BotImagem, BotAudio): 
    pass

In [ ]:
chave_api = "8834393587:AAHYjb87A856uqOfghdLUJlCn8QXbv4fB-o"
meu_bot = BotTelegram(chave_api)
meu_bot.iniciar()

Bot iniciado com sucesso! Aguardando mensagens...

image 1/1 c:\Users\heito\Documents\UFJF\Programao_Engenharia\bot_telegram\ENE140_2026_1\temp_imagem.jpg: 640x480 1 person, 104.5ms
Speed: 6.6ms preprocess, 104.5ms inference, 11.0ms postprocess per image at shape (1, 3, 640, 480)
